# Módulo 3 · Clase 2 — Gestión de archivos y permisos

**Curso de Inteligencia Artificial — Nivel básico**

---

## ¿Qué vamos a ver hoy?

| # | Bloque | Qué aprenderás |
|---|--------|----------------|
| 1 | `cp` — copiar archivos y directorios | Clonar sin mover |
| 2 | `mv` — mover y renombrar | Un comando para dos usos |
| 3 | `rm` — eliminar | Con cuidado — no hay papelera |
| 4 | `cat`, `head`, `tail`, `less` | Leer archivos sin abrirlos |
| 5 | `touch` y `nano` | Crear y editar archivos en la terminal |
| 6 | Permisos Linux (`rwx`) | Qué significa cada símbolo |
| 7 | `chmod` y `chown` | Cambiar permisos y propietario |
| 8 | Práctica | Gestionar los archivos del proyecto ML |

> ⚠️ **Regla de oro de la terminal:** `rm` no tiene papelera. Cuando borras algo, desaparece. Siempre comprueba antes de ejecutar.

---
## Preparar el entorno de demo

In [ ]:
import os, shutil
from pathlib import Path

DEMO = Path.home() / "gestion_demo"
if DEMO.exists(): shutil.rmtree(DEMO)

(DEMO / "originales").mkdir(parents=True)
(DEMO / "backups").mkdir()
(DEMO / "archivados").mkdir()

(DEMO / "originales" / "informe.txt").write_text("Informe Q1\n" * 10, encoding="utf-8")
(DEMO / "originales" / "datos.csv").write_text("id,valor\n" + "1,100\n" * 20, encoding="utf-8")
(DEMO / "originales" / "script.py").write_text("print('inicio')\n" * 5, encoding="utf-8")
(DEMO / "originales" / "temporal.tmp").write_text("borrar", encoding="utf-8")

os.chdir(DEMO)
print(f"Demo en: {DEMO}")
!ls -R

---
## 1. `cp` — Copiar

```bash
cp origen destino              # copiar un archivo
cp origen1 origen2 directorio/ # copiar varios archivos a un directorio
cp -r directorio/ destino/     # copiar directorio completo (recursivo)
cp -i origen destino           # preguntar antes de sobreescribir
cp -p origen destino           # preservar permisos y fecha
cp -u origen destino           # copiar solo si origen es más nuevo
```

El destino puede ser:
- Un **nombre de archivo** → renombra en el destino.
- Un **directorio existente** → copia dentro con el mismo nombre.
- Una **ruta nueva** → crea el archivo en esa ruta.

In [ ]:
os.chdir(DEMO)

# Copiar un archivo
!cp originales/informe.txt backups/informe_backup.txt
print("Tras copiar informe.txt:")
!ls backups/

# Copiar varios archivos a un directorio
!cp originales/datos.csv originales/script.py backups/
print("\nTras copiar varios archivos:")
!ls backups/

In [ ]:
# Copiar un directorio completo (necesita -r)
!cp -r originales/ backups/originales_copia/
print("Tras copiar directorio:")
!ls -la backups/

# Sin -r falla:
import subprocess
r = subprocess.run(["cp", "originales", "backups/intento"],
                   capture_output=True, text=True)
print(f"\nSin -r: {r.stderr.strip()}")

---
## 2. `mv` — Mover y renombrar

`mv` hace dos cosas según el destino:
- Si el destino es un **nombre diferente en el mismo directorio** → **renombra**.
- Si el destino es un **directorio** → **mueve**.

```bash
mv viejo.txt nuevo.txt         # renombrar
mv archivo.txt directorio/     # mover
mv archivo.txt dir/nuevo.txt   # mover y renombrar a la vez
mv *.log logs/                 # mover todos los .log
mv -i origen destino           # preguntar antes de sobreescribir
mv -n origen destino           # no sobreescribir nunca
```

**`mv` no tiene `-r`** — los directorios se mueven directamente.

In [ ]:
os.chdir(DEMO)

# Renombrar un archivo
!cp originales/informe.txt informe_v1.txt
!mv informe_v1.txt informe_v2.txt
print("Renombrado:")
!ls *.txt 2>/dev/null || ls informe*.txt 2>/dev/null

# Mover a otro directorio
!mv informe_v2.txt archivados/
print("\nMovido a archivados:")
!ls archivados/

# Mover y renombrar en un solo paso
!cp originales/datos.csv datos_temp.csv
!mv datos_temp.csv archivados/datos_historicos.csv
print("\nMovido y renombrado:")
!ls archivados/

---
## 3. `rm` — Eliminar

```bash
rm archivo.txt             # eliminar un archivo
rm archivo1 archivo2       # eliminar varios
rm *.tmp                   # eliminar por patrón glob
rm -r directorio/          # eliminar directorio y todo su contenido
rm -i archivo.txt          # pedir confirmación antes de borrar
rm -f archivo.txt          # forzar (no preguntar si no existe)
```

⚠️ **`rm -rf /ruta`** es uno de los comandos más peligrosos de la terminal. Borra todo de forma recursiva y forzada, sin confirmación. Nunca lo uses en directorios del sistema.

**Buenas prácticas:**
- Usa `-i` cuando no estés seguro.
- Usa `ls` con el mismo glob antes de `rm`: si `ls *.tmp` muestra lo correcto, entonces `rm *.tmp` es seguro.
- En scripts, mueve a una carpeta temporal antes de borrar definitivamente.

In [ ]:
os.chdir(DEMO)

# Verificar primero con ls (buena práctica)
print("¿Qué voy a borrar?")
!ls originales/*.tmp

# Borrar el archivo temporal
!rm originales/temporal.tmp
print("\nTras rm:")
!ls originales/

# Borrar un directorio vacío (rmdir) vs con contenido (rm -r)
!mkdir carpeta_vacia
!rmdir carpeta_vacia
print("\ncarpeta_vacia eliminada con rmdir")

# Para directorios con contenido:
!mkdir -p para_borrar/subcarpeta
!touch para_borrar/archivo.txt
print("\nAntes de rm -r:")
!ls para_borrar/

!rm -r para_borrar/
print("Tras rm -r: directorio eliminado")
!ls | grep para_borrar || echo "(no aparece — borrado correctamente)"

---
## 4. Leer archivos: `cat`, `head`, `tail`, `less`

```bash
cat archivo.txt            # mostrar todo el archivo
cat a.txt b.txt            # concatenar y mostrar dos archivos
cat -n archivo.txt         # mostrar con números de línea

head archivo.txt           # primeras 10 líneas
head -n 5 archivo.txt      # primeras 5 líneas

tail archivo.txt           # últimas 10 líneas
tail -n 20 archivo.txt     # últimas 20 líneas
tail -f log.txt            # seguir el archivo en tiempo real (logs!)

less archivo.txt           # lector interactivo (q para salir)
```

**`tail -f`** es uno de los comandos más útiles en IA: lo usarás constantemente para monitorear logs de entrenamiento en tiempo real.

In [ ]:
os.chdir(DEMO)

# Crear un archivo de log de ejemplo
Path("entrenamiento.log").write_text(
    "\n".join([
        f"Epoch {i:02d}/20 — loss: {2.0 - i*0.09:.4f} — val_loss: {2.1 - i*0.08:.4f}"
        for i in range(1, 21)
    ]) + "\n",
    encoding="utf-8"
)

print("=== head -n 5 (primeras 5 líneas) ===")
!head -n 5 entrenamiento.log

print("\n=== tail -n 5 (últimas 5 líneas) ===")
!tail -n 5 entrenamiento.log

print("\n=== cat -n (con números de línea) ===")
!cat -n entrenamiento.log

In [ ]:
# wc — contar líneas, palabras, caracteres
print("=== wc (word count) ===")
!wc entrenamiento.log
# líneas  palabras  caracteres  nombre

print("\n=== wc -l (solo líneas — muy útil) ===")
!wc -l entrenamiento.log

print("\n=== wc -l en varios archivos ===")
!wc -l originales/*.txt originales/*.csv originales/*.py

---
## 5. `touch` y `nano` — Crear y editar

```bash
touch archivo.txt          # crear archivo vacío (o actualizar fecha)
touch a.txt b.txt c.txt    # crear varios a la vez

nano archivo.txt           # editor de texto en la terminal
```

**Atajos de `nano`:**

| Atajo | Acción |
|-------|--------|
| `Ctrl+O` | Guardar |
| `Ctrl+X` | Salir |
| `Ctrl+W` | Buscar |
| `Ctrl+K` | Cortar línea |
| `Ctrl+U` | Pegar |

> Para editar archivos en servidores remotos sin interfaz gráfica, `nano` es la opción más amigable. Existe también `vim` (más potente, curva de aprendizaje pronunciada) y `emacs`.

In [ ]:
os.chdir(DEMO)

# touch: crear archivos vacíos
!touch README.md .env requirements.txt
print("Archivos creados con touch:")
!ls -la | grep -E 'README|env|requirements'

# touch también actualiza la fecha de modificación
import time
!ls -la originales/informe.txt
time.sleep(1)
!touch originales/informe.txt
print("\nTras touch (fecha actualizada):")
!ls -la originales/informe.txt

---
## 6. Permisos Linux (`rwx`)

Cada archivo en Linux tiene tres grupos de permisos:

```
-rwxr-xr--  1  ana  staff  1234  ene 15  script.py
 │││└─┬─┘└─┬─┘
 │││  │    └─ otros (rest of the world)
 │││  └─ grupo
 ││└─ propietario (owner)
 │└─ tipo: - archivo, d directorio, l enlace
 └─ (el primer carácter ya lo vimos)

Cada grupo tiene tres bits:  r  w  x
                             │  │  └─ execute (ejecutar)
                             │  └─ write (escribir)
                             └─ read (leer)
```

**Representación numérica (octal):**

| Símbolo | Octal | Significado |
|---------|-------|-------------|
| `---`   | 0     | Sin permisos |
| `--x`   | 1     | Solo ejecutar |
| `-w-`   | 2     | Solo escribir |
| `-wx`   | 3     | Escribir + ejecutar |
| `r--`   | 4     | Solo leer |
| `r-x`   | 5     | Leer + ejecutar |
| `rw-`   | 6     | Leer + escribir |
| `rwx`   | 7     | Todo |

**Permisos más comunes:**
- `644` (`rw-r--r--`) → archivo normal: dueño puede leer/escribir, el resto solo leer.
- `755` (`rwxr-xr-x`) → script ejecutable: dueño todo, el resto leer y ejecutar.
- `700` (`rwx------`) → solo el dueño puede hacer algo (claves SSH, etc.).
- `600` (`rw-------`) → archivos privados (`.env`, claves privadas).

In [ ]:
os.chdir(DEMO)

# Ver permisos actuales
print("=== Permisos actuales ===")
!ls -la originales/

# Ver en formato octal
print("\n=== Permisos en octal (stat) ===")
!stat -f "%Mp%Lp %N" originales/* 2>/dev/null || \
 stat --format="%a %n" originales/* 2>/dev/null

---
## 7. `chmod` y `chown` — Cambiar permisos

```bash
# Notación octal (más concisa)
chmod 755 script.py        # rwxr-xr-x
chmod 644 datos.csv        # rw-r--r--
chmod 600 .env             # rw------- (solo el dueño)

# Notación simbólica (más legible)
chmod +x script.py         # añadir ejecutar a todos
chmod -w archivo.txt       # quitar escritura a todos
chmod u+x script.py        # añadir ejecutar solo al usuario
chmod go-w archivo.txt     # quitar escritura a grupo y otros
chmod a=r archivo.txt      # todos solo lectura

# Recursivo
chmod -R 755 directorio/   # aplicar a todo el directorio

# Cambiar propietario
chown usuario archivo      # cambiar dueño
chown usuario:grupo archivo # cambiar dueño y grupo
```

**Regla práctica:**
- Scripts Python/Bash que vas a ejecutar → `chmod +x script.py`
- Archivos `.env` con contraseñas → `chmod 600 .env`
- Claves SSH → `chmod 700 ~/.ssh/` y `chmod 600 ~/.ssh/id_ed25519`

In [ ]:
os.chdir(DEMO)

# Hacer un script ejecutable
print("Antes de chmod:")
!ls -la originales/script.py

!chmod +x originales/script.py
print("\nTras chmod +x:")
!ls -la originales/script.py

# Proteger .env
Path(".env").write_text("API_KEY=secreto123\nDB_PASS=otra_clave\n", encoding="utf-8")
!chmod 600 .env
print("\n.env con permisos 600:")
!ls -la .env

# Ver los permisos en python
import stat
modo = Path("originales/script.py").stat().st_mode
print(f"\nPermisos de script.py en octal: {oct(modo)[-3:]}")
print(f"Permisos en simbólico: {stat.filemode(modo)}")

---
## 8. Práctica — Gestionar los archivos del proyecto ML

---

### Ejercicio 1 — Organizar un directorio caótico

**Enunciado:** Tienes un directorio `caos/` con archivos mezclados. Organízalos automáticamente:

1. Crea `~/caos/` con estos archivos: `modelo.pkl`, `entrenamiento.log`, `datos.csv`, `config.yaml`, `notas.txt`, `backup.tar.gz`, `script_test.py`, `reporte.pdf`, `imagen.png`, `README.md`.
2. Escribe un script Python que los organice en subcarpetas por tipo:
   - `modelos/` → `.pkl`, `.h5`, `.pt`
   - `datos/` → `.csv`, `.json`, `.parquet`
   - `config/` → `.yaml`, `.yml`, `.toml`, `.ini`
   - `logs/` → `.log`
   - `scripts/` → `.py`, `.sh`
   - `docs/` → `.txt`, `.md`, `.pdf`
   - `otros/` → el resto
3. Verifica con `ls` cada subcarpeta.

In [ ]:
# ✏️ TU CÓDIGO AQUÍ
from pathlib import Path
import shutil, os


<details>
<summary>💡 Ver solución — Ejercicio 1</summary>

```python
from pathlib import Path
import shutil, os

CAOS = Path.home() / "caos"
CAOS.mkdir(exist_ok=True)

# Crear archivos de ejemplo
for nombre in ["modelo.pkl", "entrenamiento.log", "datos.csv", "config.yaml",
               "notas.txt", "backup.tar.gz", "script_test.py", "reporte.pdf",
               "imagen.png", "README.md"]:
    (CAOS / nombre).write_text(f"contenido de {nombre}", encoding="utf-8")

# Mapa extensión → subcarpeta
CATEGORIAS = {
    "modelos":  {".pkl", ".h5", ".pt", ".onnx"},
    "datos":    {".csv", ".json", ".parquet", ".tsv"},
    "config":   {".yaml", ".yml", ".toml", ".ini", ".cfg"},
    "logs":     {".log"},
    "scripts":  {".py", ".sh", ".bash"},
    "docs":     {".txt", ".md", ".pdf", ".rst"},
}

def categorizar(ext):
    for categoria, extensiones in CATEGORIAS.items():
        if ext in extensiones:
            return categoria
    return "otros"

movidos = 0
for archivo in CAOS.iterdir():
    if not archivo.is_file():
        continue
    categoria = categorizar(archivo.suffix.lower())
    destino = CAOS / categoria
    destino.mkdir(exist_ok=True)
    shutil.move(str(archivo), destino / archivo.name)
    print(f"  {archivo.name} → {categoria}/")
    movidos += 1

print(f"\nTotal movidos: {movidos}")

os.chdir(CAOS)
!ls -la
```
</details>

---
### Ejercicio 2 — Copiar y versionar archivos

**Enunciado:** Practica el ciclo de versionado manual con `cp` y `mv`:

1. En `~/ml_proyecto/`, crea `modelo.py` con una función `predecir(x)`.
2. Antes de modificarlo, crea un backup: `modelo_v1.py`.
3. Modifica `modelo.py`: añade una función `evaluar(x, y)` en el original.
4. Antes de otra modificación, crea `modelo_v2.py`.
5. Mueve los backups a `modelos/checkpoints/`.
6. Lista el directorio `checkpoints/` para verificar.

In [ ]:
# ✏️ TU CÓDIGO AQUÍ


<details>
<summary>💡 Ver solución — Ejercicio 2</summary>

```python
from pathlib import Path
import os

ML = Path.home() / "ml_proyecto"
os.chdir(ML)

# 1. Crear modelo.py inicial
Path("modelo.py").write_text("""\
def predecir(x):
    return x * 2.0
""", encoding="utf-8")

# 2. Backup antes de modificar
!cp modelo.py modelo_v1.py
print("Backup v1 creado")

# 3. Modificar el original
contenido = Path("modelo.py").read_text()
Path("modelo.py").write_text(contenido + """
def evaluar(x, y):
    pred = predecir(x)
    return abs(pred - y)
""", encoding="utf-8")
print("modelo.py modificado")

# 4. Otro backup
!cp modelo.py modelo_v2.py
print("Backup v2 creado")

# 5. Mover backups a checkpoints
!mv modelo_v1.py modelo_v2.py modelos/checkpoints/

# 6. Verificar
print("\nCheckpoints:")
!ls -la modelos/checkpoints/
```

**Nota:** En la práctica esto se hace con Git (Módulo 2), pero entender `cp`/`mv` sigue siendo esencial para scripts de automatización.
</details>

---
### Ejercicio 3 — Leer y analizar logs

**Enunciado:** Crea un log de entrenamiento realista y analízalo:

1. Genera `entrenamiento.log` con 50 épocas: `Epoch XX/50 — loss: Y.YYYY — val_loss: Z.ZZZZ — accuracy: 0.XX`.
2. Muestra las primeras cinco y las últimas cinco líneas.
3. Muestra solo las líneas donde la accuracy supera 0.90 (usa `grep`).
4. Escribe una función Python `resumir_log(ruta)` que lea el archivo y devuelva: época con menor loss, época con mayor accuracy, y si el modelo mejoró (loss final < loss inicial).

In [ ]:
# ✏️ TU CÓDIGO AQUÍ


<details>
<summary>💡 Ver solución — Ejercicio 3</summary>

```python
import os, re
from pathlib import Path

ML = Path.home() / "ml_proyecto"
os.chdir(ML)

# 1. Generar log de 50 épocas
import math
lineas = []
for i in range(1, 51):
    loss = 2.0 * math.exp(-i * 0.07)
    val_loss = loss * 1.05 + 0.01
    accuracy = 1 - loss / 2.5
    lineas.append(
        f"Epoch {i:02d}/50 — loss: {loss:.4f} — val_loss: {val_loss:.4f} — accuracy: {accuracy:.4f}"
    )
Path("logs/entrenamiento.log").write_text("\n".join(lineas) + "\n", encoding="utf-8")

# 2. head y tail
print("=== Primeras 5 ===")
!head -n 5 logs/entrenamiento.log
print("\n=== Últimas 5 ===")
!tail -n 5 logs/entrenamiento.log

# 3. Épocas con accuracy > 0.90
print("\n=== Accuracy > 0.90 ===")
!grep -E 'accuracy: 0\.9[0-9]' logs/entrenamiento.log

# 4. Función Python
def resumir_log(ruta):
    patron = re.compile(
        r'Epoch (\d+)/\d+ — loss: ([\d.]+) — val_loss: [\d.]+ — accuracy: ([\d.]+)'
    )
    epocas = []
    for linea in Path(ruta).read_text(encoding="utf-8").splitlines():
        m = patron.search(linea)
        if m:
            epocas.append({
                "epoca": int(m.group(1)),
                "loss": float(m.group(2)),
                "accuracy": float(m.group(3)),
            })
    if not epocas:
        print("No se encontraron datos")
        return

    mejor_loss = min(epocas, key=lambda e: e["loss"])
    mejor_acc  = max(epocas, key=lambda e: e["accuracy"])
    mejoro = epocas[-1]["loss"] < epocas[0]["loss"]

    print(f"\n  Épocas analizadas:   {len(epocas)}")
    print(f"  Menor loss:          Epoch {mejor_loss['epoca']:02d} → {mejor_loss['loss']:.4f}")
    print(f"  Mayor accuracy:      Epoch {mejor_acc['epoca']:02d} → {mejor_acc['accuracy']:.4f}")
    print(f"  ¿Modelo mejoró?      {'Sí ✅' if mejoro else 'No ⚠️'}")
    print(f"  Loss inicial→final:  {epocas[0]['loss']:.4f} → {epocas[-1]['loss']:.4f}")

resumir_log("logs/entrenamiento.log")
```
</details>

---
### Ejercicio 4 — Gestión de permisos

**Enunciado:** Aplica los permisos correctos a los archivos del proyecto:

1. En `~/ml_proyecto/`, haz todos los archivos `.py` ejecutables con un solo comando.
2. Protege el archivo `.env` con permisos `600`.
3. Haz que todos los archivos `.csv` sean de solo lectura para el grupo y otros (sin cambiar los del dueño).
4. Escribe una función Python `auditar_permisos(ruta)` que liste todos los archivos y marque con ⚠️ los que tienen permisos peligrosos (escritura para otros: `o+w`).

In [ ]:
# ✏️ TU CÓDIGO AQUÍ


---
### Ejercicio 5 (reto) — Script de backup automático

**Enunciado:** Escribe la función Python `backup(origen, destino)` que:

1. Copia el directorio `origen` a `destino/backup_YYYYMMDD_HHMMSS/` (con fecha y hora en el nombre).
2. Excluye archivos con extensión `.tmp`, `.log`, `.pyc` y directorios `__pycache__`.
3. Aplica permisos `644` a todos los archivos copiados y `755` a los directorios.
4. Genera un archivo `MANIFEST.txt` en la carpeta de backup con: fecha, origen, número de archivos copiados, lista de archivos excluidos.
5. Devuelve la ruta del backup creado.

Pruébala haciendo backup de `~/ml_proyecto` en `~/backups`.

In [ ]:
# ✏️ TU CÓDIGO AQUÍ
from pathlib import Path
from datetime import datetime
import shutil, stat


<details>
<summary>💡 Ver solución — Ejercicio 5</summary>

```python
from pathlib import Path
from datetime import datetime
import shutil, stat, os

EXCLUIR_EXT = {".tmp", ".log", ".pyc"}
EXCLUIR_DIRS = {"__pycache__", ".git", ".venv", "venv"}

def backup(origen, destino):
    origen  = Path(origen).expanduser().resolve()
    destino = Path(destino).expanduser().resolve()
    destino.mkdir(parents=True, exist_ok=True)

    marca = datetime.now().strftime("%Y%m%d_%H%M%S")
    carpeta_backup = destino / f"backup_{marca}"
    carpeta_backup.mkdir()

    copiados = []
    excluidos = []

    for src in origen.rglob("*"):
        rel = src.relative_to(origen)

        # Excluir directorios prohibidos
        if any(parte in EXCLUIR_DIRS for parte in rel.parts):
            continue

        dst = carpeta_backup / rel

        if src.is_dir():
            dst.mkdir(parents=True, exist_ok=True)
            dst.chmod(0o755)
        elif src.is_file():
            if src.suffix.lower() in EXCLUIR_EXT:
                excluidos.append(str(rel))
                continue
            dst.parent.mkdir(parents=True, exist_ok=True)
            shutil.copy2(src, dst)
            dst.chmod(0o644)
            copiados.append(str(rel))

    # Manifiesto
    manifiesto = [
        f"Backup creado: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}",
        f"Origen:        {origen}",
        f"Destino:       {carpeta_backup}",
        f"Archivos copiados: {len(copiados)}",
        f"Archivos excluidos: {len(excluidos)}",
        "",
        "=== ARCHIVOS COPIADOS ===",
    ] + copiados + [
        "",
        "=== ARCHIVOS EXCLUIDOS ===",
    ] + excluidos

    (carpeta_backup / "MANIFEST.txt").write_text(
        "\n".join(manifiesto), encoding="utf-8"
    )

    print(f"✅ Backup creado: {carpeta_backup}")
    print(f"   Archivos copiados:  {len(copiados)}")
    print(f"   Archivos excluidos: {len(excluidos)}")
    return carpeta_backup


ruta = backup("~/ml_proyecto", "~/backups")
print("\nContenido del backup:")
import subprocess
subprocess.run(["ls", "-la", str(ruta)])
```
</details>

---
## Resumen de la clase

```bash
# ─── Copiar ────────────────────────────────────────────────
cp origen destino          # copiar archivo
cp -r dir/ destino/        # copiar directorio
cp -i origen destino       # preguntar antes de sobreescribir

# ─── Mover / Renombrar ─────────────────────────────────────
mv viejo.txt nuevo.txt     # renombrar
mv archivo.txt dir/        # mover
mv archivo.txt dir/nuevo   # mover y renombrar

# ─── Eliminar ──────────────────────────────────────────────
rm archivo.txt             # borrar archivo
rm -r directorio/          # borrar directorio
rm -i archivo.txt          # con confirmación

# ─── Leer ──────────────────────────────────────────────────
cat archivo.txt            # mostrar todo
head -n 5 archivo          # primeras 5 líneas
tail -n 5 archivo          # últimas 5 líneas
tail -f log.txt            # seguir en tiempo real
wc -l archivo              # contar líneas

# ─── Crear / Editar ────────────────────────────────────────
touch archivo.txt          # crear vacío / actualizar fecha
nano archivo.txt           # editor en terminal

# ─── Permisos ──────────────────────────────────────────────
chmod 755 script.py        # rwxr-xr-x
chmod 644 datos.csv        # rw-r--r--
chmod 600 .env             # rw------- (privado)
chmod +x script.py         # añadir ejecutar
chmod -R 755 directorio/   # recursivo
```

## ¿Qué viene en la próxima clase?

En la **Clase 3** entramos en las **herramientas de productividad**: `grep` para buscar texto, `find` para buscar archivos, pipes `|` para encadenar comandos y redirecciones `>` para capturar resultados.